# Cuaderno U1-02. Tipos de modelos, conceptuales, matematicos, estadisticos y computacionales

**Modelacion y Simulacion Computacional** · Maestria en Ingenieria · Universidad de Sucre

Unidad 1, Fundamentos de modelacion en ingenieria · Subtema 1.2 del plan de asignatura

Docente Daniel David Otero Meza · Periodo 2026-2

<!-- ENLACE_COLAB -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msc-unisucre/msc2026-material/blob/main/03_cuadernos/Unidad1/U1_02_tipos_de_modelos.ipynb)

*El docente reemplaza `msc-unisucre/msc2026-material` por la direccion real del repositorio del
curso antes de publicar el cuaderno.*

Este cuaderno acompana la Seccion 1.2 del libro. Recorre los cuatro estratos de la construccion de un modelo, aplica los seis criterios transversales de la Tabla 1.1 y reproduce integramente el Ejemplo 1.1, que resuelve una misma camara de contacto de cloro con tres modelos de complejidad creciente y muestra que la eleccion del estrato se paga en reactivo.

## Objetivos de aprendizaje

Al terminar este cuaderno el estudiante estara en capacidad de hacer lo siguiente.

1. Situar un modelo en los cuatro estratos, conceptual, matematico, estadistico y computacional, y explicar que aporta cada uno.
2. Clasificar un modelo segun los seis criterios transversales de la Tabla 1.1 y anticipar la consecuencia numerica de cada posicion.
3. Reproducir los tres modelos del Ejemplo 1.1 y verificar sus resultados contra la solucion analitica del reactor de flujo disperso.
4. Estimar el orden observado de un esquema numerico por refinamiento de la malla, como pide el Problema 1-24.

## Puesta a punto

La primera celda detecta el entorno e instala unicamente lo que falte. La segunda fija la semilla del curso y la paleta del libro. La tercera define las funciones de verificacion que se usan mas abajo. Ejecutelas en orden antes de continuar.

In [ ]:
# Puesta a punto del entorno. Detecta Colab e instala solo lo que falte.
import importlib
import importlib.util
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes):
    """Instala los paquetes ausentes sin reinstalar los que ya estan."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes],
                       check=True)
    return faltantes


AUSENTES = asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
                     "matplotlib": "matplotlib", "sympy": "sympy"})

print("Entorno de ejecucion:", "Google Colab" if EN_COLAB else "JupyterLab local")
print("Paquetes instalados en esta sesion:", AUSENTES or "ninguno, ya estaban")

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sympy as sp

# Semilla unica de la asignatura. Ningun resultado depende de una corrida.
SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

# Paleta del libro. Los cuadernos usan los mismos colores que las figuras.
PALETA = {
    "azul": "#1F4E79",
    "rojo": "#B3251E",
    "verde": "#2E7D32",
    "naranja": "#E07B00",
    "gris": "#5A5A5A",
    "morado": "#6A3D9A",
}

plt.rcParams.update({
    "figure.figsize": (8.6, 4.6),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=list(PALETA.values())),
    "font.size": 10.0,
    "legend.frameon": True,
    "legend.framealpha": 0.92,
})

print(f"NumPy {np.__version__} · SciPy {scipy.__version__} · pandas {pd.__version__}")
print(f"SymPy {sp.__version__} · semilla del curso {SEMILLA}")

In [ ]:
# Bandera de revision de los ejercicios guiados.
# Mientras valga False el cuaderno se ejecuta completo aunque falten celdas.
# Pongala en True cuando haya completado las celdas marcadas para completar.
REVISAR = False
print("REVISAR =", REVISAR)

In [ ]:
def verificar_libro(nombre, obtenido, publicado, tolerancia=1e-3, unidad=""):
    """Contrasta un resultado calculado con la cifra que publica el libro."""
    valor = float(obtenido)
    escala = abs(publicado) if publicado else 1.0
    error = abs(valor - publicado) / escala
    print(f"{nombre:<46s} calculado {valor:>12.6g} {unidad:<10s}"
          f" libro {publicado:>12.6g}  error rel. {error:.1e}")
    assert error <= tolerancia, f"{nombre} se aparta de la cifra publicada"
    return valor


def comprobar(nombre, obtenido, referencia, tolerancia=1e-3, unidad=""):
    """Revisa una celda de ejercicio contra su valor de referencia.

    Con REVISAR en False solo informa que el ejercicio sigue pendiente, de modo
    que el cuaderno nunca se detiene por una celda sin completar.
    """
    if not REVISAR:
        print(f"[pendiente]  {nombre}")
        return False
    valor = float(obtenido)
    escala = abs(referencia) if referencia else 1.0
    error = abs(valor - referencia) / escala
    marca = "correcto " if error <= tolerancia else "revisar  "
    print(f"[{marca}]  {nombre} = {valor:.6g} {unidad}"
          f"  referencia {referencia:.6g}  error rel. {error:.2e}")
    assert error <= tolerancia, f"{nombre} no coincide con la referencia"
    return True


def ruta_datos(nombre):
    """Ubica un archivo de la carpeta datos sin usar rutas absolutas.

    Funciona igual en Colab, donde el cuaderno suele abrirse en el directorio
    de trabajo, y en una copia local del repositorio, donde el cuaderno vive
    dentro de Unidad1 o de soluciones.
    """
    candidatas = (Path("datos"),
                  Path("..") / "datos",
                  Path("..") / ".." / "datos",
                  Path("03_cuadernos") / "datos")
    for base in candidatas:
        if (base / nombre).exists():
            return base / nombre
    for base in candidatas:        # la carpeta existe pero el archivo aun no
        if base.is_dir():
            return base / nombre
    return Path("datos") / nombre  # entorno nuevo, como una sesion de Colab


def cargar_o_generar(nombre, generador):
    """Lee el archivo de datos y, si no esta, lo reconstruye con la semilla."""
    ruta = ruta_datos(nombre)
    if ruta.exists():
        print(f"Datos leidos de {ruta}")
        return pd.read_csv(ruta)
    tabla = generador()
    ruta.parent.mkdir(parents=True, exist_ok=True)
    tabla.to_csv(ruta, index=False)
    print(f"Datos regenerados con la semilla {SEMILLA} y guardados en {ruta}")
    return tabla


def integrar_trapecio(valores, muestras):
    """Regla del trapecio compatible con NumPy 1 y con NumPy 2."""
    regla = getattr(np, "trapezoid", None) or np.trapz
    return float(regla(valores, muestras))


print("Funciones auxiliares disponibles.")

## 1. Los cuatro estratos

El libro ordena la variedad de objetos que reciben el nombre de modelo en cuatro
estratos sucesivos, y la palabra clave es sucesivos, porque cada estrato
presupone el anterior y lo hace operativo.

El modelo conceptual, de la Definicion 1.3, fija la frontera, enumera los
componentes que se retienen, identifica los flujos y declara las hipotesis, todo
antes de escribir ninguna ecuacion. El matematico, de la Definicion 1.4, es la
terna de sistema, pregunta y enunciados que traduce esa descripcion en balances
y ecuaciones constitutivas. El estadistico, de la Definicion 1.5, incorpora la
componente aleatoria e infiere de los datos la parte del comportamiento que no
sale de un principio de conservacion. La Figura 1.3 del libro encadena los
cuatro estratos y anade la dimension del origen de las relaciones, mecanicista,
empirico o hibrido. El computacional, de la Definicion 1.8,
convierte lo anterior en un programa que lo resuelve sobre una malla y con una
aritmetica finitas, y por eso anade un error de discretizacion y uno de redondeo
que le son propios.

La celda siguiente escribe el modelo conceptual de la camara de contacto de
cloro del Ejemplo 1.1, que es el entregable que con mas frecuencia se omite.

In [ ]:
MODELO_CONCEPTUAL = [
    ("frontera", "las paredes de la camara, con una seccion de entrada y una de salida"),
    ("componentes retenidos", "el agua y el desinfectante residual disuelto"),
    ("flujos", "adveccion por el caudal permanente y dispersion longitudinal"),
    ("procesos", "decaimiento del desinfectante con cinetica de primer orden"),
    ("hipotesis 1", "caudal, volumen y temperatura constantes"),
    ("hipotesis 2", "constante de decaimiento conocida e independiente de la posicion"),
    ("hipotesis 3", "dispersion longitudinal estimada de la geometria de la camara"),
    ("hipotesis 4", "la camara arranca limpia, con concentracion nula en su interior"),
    ("hipotesis 5", "sin cortocircuitos ni zonas muertas adicionales al termino dispersivo"),
]
conceptual = pd.DataFrame(MODELO_CONCEPTUAL, columns=["elemento", "declaracion"])
conceptual.set_index("elemento")

## 2. Mecanicista frente a empirico

La Definicion 1.6 llama mecanicista al modelo cuyas ecuaciones se derivan de
principios de conservacion, con parametros de significado fisico propio, y la
Definicion 1.7 llama empirico al que elige su estructura por conveniencia de
ajuste. Lo determinante no es la forma sino el origen, porque de el depende la
capacidad de extrapolar.

La celda siguiente hace visible esa diferencia sobre el decaimiento del
desinfectante. El modelo mecanicista es la exponencial que sale del balance con
cinetica de primer orden. El empirico es una recta ajustada a las mismas
observaciones dentro de una ventana estrecha de tiempo. Dentro de la ventana los
dos son excelentes, y fuera de ella solo uno sigue teniendo sentido fisico. El
tratamiento completo de este asunto ocupa el cuaderno U1-05.

In [ ]:
CONSTANTE_DECAIMIENTO = 0.35    # 1/h

tiempos_medidos = np.linspace(0.5, 2.0, 8)                    # h
observado = (np.exp(-CONSTANTE_DECAIMIENTO * tiempos_medidos)
             + rng.normal(0.0, 0.004, tiempos_medidos.size))
pendiente, corte = np.polyfit(tiempos_medidos, observado, 1)

tiempos = np.linspace(0.0, 8.0, 300)
mecanicista = np.exp(-CONSTANTE_DECAIMIENTO * tiempos)
empirico = pendiente * tiempos + corte

figura, eje = plt.subplots()
eje.plot(tiempos, mecanicista, color=PALETA["azul"], lw=1.9,
         label="Mecanicista, sale del balance")
eje.plot(tiempos, empirico, color=PALETA["rojo"], lw=1.7, ls="--",
         label="Empirico, recta ajustada")
eje.plot(tiempos_medidos, observado, "o", color=PALETA["gris"], ms=5,
         label="Ventana de observacion")
eje.axvspan(0.5, 2.0, color=PALETA["azul"], alpha=0.10, lw=0)
eje.axhline(0.0, color=PALETA["gris"], lw=0.8)
eje.set_xlabel("Tiempo de contacto (h)")
eje.set_ylabel("Concentracion relativa remanente (adimensional)")
eje.set_ylim(-0.6, 1.05)
eje.legend(loc="upper right")
plt.show()

cruce_cero = -corte / pendiente
print(f"La recta ajustada cruza el cero a las {cruce_cero:.2f} h y luego "
      "predice concentraciones negativas.")
print("El modelo mecanicista tiende a cero sin cruzarlo, porque el balance del "
      "que sale no admite masa negativa.")
assert cruce_cero < 8.0

## 3. Los seis criterios transversales

La Tabla 1.1 del libro recoge seis criterios que no compiten entre si sino que
se aplican de forma simultanea, de modo que un modelo no pertenece a una
categoria sino que ocupa un punto en un espacio de seis decisiones. La Figura 1.4
los representa como seis ejes independientes.

In [ ]:
CRITERIOS = [
    {"criterio": "origen", "polos": "mecanicista, empirico",
     "consecuencia practica": "define si la extrapolacion tiene respaldo",
     "ejemplo": "secador con balance de energia frente a curva ajustada"},
    {"criterio": "tiempo", "polos": "estacionario, dinamico",
     "consecuencia practica": "sistema algebraico frente a integracion",
     "ejemplo": "tirante normal frente a transito de creciente"},
    {"criterio": "aleatoriedad", "polos": "deterministico, estocastico",
     "consecuencia practica": "una corrida frente a un conjunto de realizaciones",
     "ejemplo": "demanda media frente a curva de duracion de carga"},
    {"criterio": "espacio", "polos": "concentrados, distribuidos",
     "consecuencia practica": "ecuaciones ordinarias frente a derivadas parciales",
     "ejemplo": "tanque agitado frente a columna de percolacion"},
    {"criterio": "linealidad", "polos": "lineal, no lineal",
     "consecuencia practica": "superposicion y solucion analitica de referencia",
     "ejemplo": "pozos en acuifero confinado frente a acuifero libre"},
    {"criterio": "estado", "polos": "continuo, discreto",
     "consecuencia practica": "integrador numerico frente a planificador de eventos",
     "ejemplo": "cuarto frio frente a fallas de una red de distribucion"},
]
tabla_criterios = pd.DataFrame(CRITERIOS).set_index("criterio")

CRITERIOS_VALIDOS = {
    "origen": ("mecanicista", "empirico", "hibrido"),
    "tiempo": ("estacionario", "dinamico"),
    "aleatoriedad": ("deterministico", "estocastico"),
    "espacio": ("concentrados", "distribuidos"),
    "linealidad": ("lineal", "no lineal"),
    "estado": ("continuo", "discreto", "hibrido"),
}


def clasificar(**posiciones):
    """Valida la clasificacion de un modelo segun los seis criterios."""
    faltantes = [c for c in CRITERIOS_VALIDOS if c not in posiciones]
    invalidos = {c: v for c, v in posiciones.items()
                 if c in CRITERIOS_VALIDOS and v not in CRITERIOS_VALIDOS[c]}
    if faltantes:
        raise ValueError(f"faltan criterios por responder, {faltantes}")
    if invalidos:
        raise ValueError(f"posiciones no admitidas, {invalidos}")
    return pd.Series(posiciones, name="posicion")


tabla_criterios

## 4. Ejemplo 1.1 del libro, tres modelos de una camara de contacto

Una camara de contacto rectangular tiene 30 m de longitud y una seccion
transversal de 4 m2, de modo que su volumen util es de 120 m3. Recibe un caudal
permanente de 40 m3/h y el desinfectante residual decae con cinetica de primer
orden y constante de 0.35 por hora, con un coeficiente de dispersion
longitudinal de 25 m2/h.

Los tres modelos comparten dos grupos adimensionales, que son el numero de
Peclet y el numero de Damkohler. El libro reporta un tiempo de residencia de
3 h, un Peclet de 12 y un Damkohler de 1.05.

In [ ]:
CAUDAL = 40.0            # m3/h
SECCION = 4.0            # m2
LARGO = 30.0             # m
DECAIMIENTO = 0.35       # 1/h
DISPERSION = 25.0        # m2/h

VOLUMEN = SECCION * LARGO            # m3
VELOCIDAD = CAUDAL / SECCION         # m/h
RESIDENCIA = VOLUMEN / CAUDAL        # h
PECLET = VELOCIDAD * LARGO / DISPERSION       # adimensional
DAMKOHLER = DECAIMIENTO * RESIDENCIA          # adimensional

verificar_libro("volumen util", VOLUMEN, 120, 1e-6, "m3")
verificar_libro("tiempo de residencia", RESIDENCIA, 3.0, 1e-6, "h")
verificar_libro("numero de Peclet", PECLET, 12.0, 1e-6)
verificar_libro("numero de Damkohler", DAMKOHLER, 1.05, 1e-6)

### Los tres modelos, en forma comparable

El Listado 1.1 del libro escribe las tres formulaciones como funciones que
reciben la misma informacion fisica y devuelven respuestas distintas, sin que
ninguna sea incorrecta. El primero supone mezcla completa y estado estacionario,
con lo cual el balance se reduce a una relacion algebraica. El segundo mantiene
la mezcla completa pero admite acumulacion, y produce una ecuacion diferencial
ordinaria. El tercero resuelve el gradiente longitudinal de la Ecuacion 1.2 por
el metodo de lineas.

La condicion de entrada es la de Danckwerts, que iguala el flujo advectivo
entrante al flujo total en la seccion de ingreso, y por eso la cara de entrada
no lleva termino dispersivo. Esa eleccion es una decision de modelacion por
derecho propio, no un detalle de implementacion.

In [ ]:
from scipy.integrate import solve_ivp


def estacionario_mezclado():
    """Modelo 1. Mezcla completa en estado estacionario, C/C0 adimensional."""
    return 1.0 / (1.0 + DECAIMIENTO * RESIDENCIA)


def dinamico_mezclado(t):
    """Modelo 2. Mezcla completa con acumulacion, C/C0 adimensional."""
    constante = 1.0 / RESIDENCIA + DECAIMIENTO      # 1/h
    return estacionario_mezclado() * (1.0 - np.exp(-constante * np.asarray(t)))


def distribuido(t_final=12.0, nodos=600, malla_tiempo=None):
    """Modelo 3. Adveccion, dispersion y reaccion por el metodo de lineas."""
    dx = LARGO / nodos
    velocidad = VELOCIDAD

    def derivada(_t, c):
        aguas_arriba = np.empty(nodos + 1)
        aguas_arriba[0] = 1.0                       # concentracion de entrada
        aguas_arriba[1:] = c
        flujo = velocidad * aguas_arriba            # adveccion aguas arriba
        flujo[1:nodos] += -DISPERSION * (c[1:] - c[:-1]) / dx   # dispersion interna
        flujo[nodos] = velocidad * c[-1]            # salida sin gradiente
        return -(flujo[1:] - flujo[:-1]) / dx - DECAIMIENTO * c

    solucion = solve_ivp(derivada, (0.0, t_final), np.zeros(nodos),
                         method="BDF", rtol=1e-8, atol=1e-10,
                         t_eval=malla_tiempo)
    return solucion.t, solucion.y[-1, :]


def wehner_wilhelm(peclet, damkohler):
    """Solucion analitica del reactor de flujo disperso de primer orden."""
    a = np.sqrt(1.0 + 4.0 * damkohler / peclet)
    numerador = 4.0 * a * np.exp(peclet / 2.0)
    denominador = ((1.0 + a)**2 * np.exp(a * peclet / 2.0)
                   - (1.0 - a)**2 * np.exp(-a * peclet / 2.0))
    return float(numerador / denominador)


constante_tiempo = 1.0 / (1.0 / RESIDENCIA + DECAIMIENTO)      # h
malla = np.linspace(0.0, 12.0, 481)
tiempo_num, salida_num = distribuido(malla_tiempo=malla)

verificar_libro("modelo 1, C/C0 de mezcla completa", estacionario_mezclado(), 0.488, 1e-3)
verificar_libro("modelo 2, constante de tiempo", constante_tiempo, 1.463, 1e-3, "h")
verificar_libro("solucion analitica del reactor disperso",
                wehner_wilhelm(PECLET, DAMKOHLER), 0.376441, 1e-5)
verificar_libro("modelo 3 con 600 celdas", salida_num[-1], 0.376659, 1e-4)

### Verificacion del Ejemplo 1.1

El libro reporta que la solucion analitica del reactor disperso vale 0.376441,
que con 600 celdas el esquema numerico entrega 0.376659 y que el error relativo
es de 5.8e-4. Reporta ademas que los dos modelos de mezcla completa sobrestiman
la concentracion residual en un 29.6 por ciento respecto del distribuido.

In [ ]:
analitica = wehner_wilhelm(PECLET, DAMKOHLER)
error_600 = abs(float(salida_num[-1]) - analitica) / analitica
sobrestimacion = 100.0 * (estacionario_mezclado() / analitica - 1.0)

verificar_libro("error relativo con 600 celdas", error_600, 5.8e-4, 2e-2)
verificar_libro("sobrestimacion de la mezcla completa", sobrestimacion, 29.6, 1e-3, "%")

figura, eje = plt.subplots(figsize=(8.6, 5.0))
eje.axhline(estacionario_mezclado(), color=PALETA["gris"], lw=1.6, ls=(0, (5, 3)),
            label="Modelo 1, algebraico estacionario de mezcla completa")
eje.plot(malla, dinamico_mezclado(malla), color=PALETA["azul"], lw=1.8,
         label="Modelo 2, dinamico de parametros concentrados")
eje.plot(tiempo_num, salida_num, color=PALETA["rojo"], lw=2.0,
         label="Modelo 3, dinamico de parametros distribuidos")
eje.axhline(analitica, color=PALETA["rojo"], lw=0.9, ls=(0, (1, 2)))
eje.axvline(RESIDENCIA, color=PALETA["naranja"], lw=1.0, ls=(0, (2, 2)))
eje.annotate(f"C/C0 = {estacionario_mezclado():.3f}", (11.8, estacionario_mezclado() + 0.02),
             ha="right", color=PALETA["gris"], fontsize=9)
eje.annotate(f"C/C0 = {analitica:.3f}", (11.8, analitica - 0.03),
             ha="right", va="top", color=PALETA["rojo"], fontsize=9)
eje.annotate("tiempo de residencia", (RESIDENCIA, 0.06), xytext=(4.6, 0.16),
             color=PALETA["naranja"], fontsize=9,
             arrowprops=dict(arrowstyle="-|>", color=PALETA["naranja"], lw=0.7))
eje.set_xlim(0.0, 12.0)
eje.set_ylim(0.0, 1.0)
eje.set_xlabel("Tiempo transcurrido (h)")
eje.set_ylabel("Concentracion relativa a la salida, C/C0 (adimensional)")
eje.legend(loc="upper left", fontsize=8.8)
eje.set_title("Ejemplo 1.1 del libro, Figura 1.5")
plt.show()

print(f"Los dos modelos de mezcla completa sobrestiman el residual en "
      f"{sobrestimacion:.1f} por ciento. Para verificar un residual minimo esa "
      "sobrestimacion es conservadora, y para dosificar cloro es antieconomica.")

### La clasificacion de los tres modelos

Aplicar los seis criterios a las tres formulaciones deja a la vista que la
diferencia no esta en la dificultad del calculo sino en el modelo conceptual que
cada una encarna.

In [ ]:
clasificaciones = pd.DataFrame({
    "modelo 1": clasificar(origen="mecanicista", tiempo="estacionario",
                           aleatoriedad="deterministico", espacio="concentrados",
                           linealidad="lineal", estado="continuo"),
    "modelo 2": clasificar(origen="mecanicista", tiempo="dinamico",
                           aleatoriedad="deterministico", espacio="concentrados",
                           linealidad="lineal", estado="continuo"),
    "modelo 3": clasificar(origen="mecanicista", tiempo="dinamico",
                           aleatoriedad="deterministico", espacio="distribuidos",
                           linealidad="lineal", estado="continuo"),
})
clasificaciones

## 5. Ejercicios guiados

Cinco celdas incompletas, cada una con la marca `# COMPLETE:`, un valor de
partida deliberadamente incorrecto y su verificacion inmediata. Cuando las
termine, ponga `REVISAR = True` en la celda de la bandera y ejecute todo de
nuevo.

### Ejercicio 1. Los dos grupos adimensionales

Una segunda camara del mismo tren tiene 24 m de longitud, seccion de 3 m2,
caudal de 36 m3/h, constante de decaimiento de 0.42 por hora y dispersion
longitudinal de 18 m2/h.

In [ ]:
# COMPLETE: escriba la funcion que devuelve el numero de Peclet y el numero de
# Damkohler de una camara, definidos como uL/D_L y k por el tiempo de
# residencia, y evaluela para la segunda camara del tren.
def grupos_camara(caudal, seccion, largo, decaimiento, dispersion):
    return float("nan"), float("nan")   # valores de partida incorrectos


peclet_dos, damkohler_dos = grupos_camara(36.0, 3.0, 24.0, 0.42, 18.0)

In [ ]:
uno = comprobar("Peclet de la segunda camara", peclet_dos, 16.0, 1e-6)
dos = comprobar("Damkohler de la segunda camara", damkohler_dos, 0.84, 1e-6)

### Ejercicio 2. Modelo estacionario de mezcla completa

In [ ]:
# COMPLETE: escriba la concentracion relativa de salida del reactor de mezcla
# completa en estado estacionario, que vale 1 sobre 1 mas el numero de
# Damkohler, y evaluela para la segunda camara.
def mezcla_completa(damkohler):
    return float("nan")   # valor de partida deliberadamente incorrecto


salida_mezcla_dos = mezcla_completa(damkohler_dos)

In [ ]:
uno = comprobar("C/C0 de la segunda camara en mezcla completa",
                salida_mezcla_dos, 0.5434783, 1e-5)
dos = comprobar("C/C0 de la camara del Ejemplo 1.1",
                mezcla_completa(DAMKOHLER), 0.488, 1e-3)

### Ejercicio 3. Distancia entre el modelo concentrado y el distribuido

Con el Peclet y el Damkohler de la segunda camara, compare el modelo de mezcla
completa con la solucion analitica del reactor de flujo disperso y diga en que
porcentaje lo sobrestima.

In [ ]:
# COMPLETE: evalue la funcion wehner_wilhelm para la segunda camara y calcule la
# sobrestimacion porcentual del modelo de mezcla completa respecto de ella.
salida_disperso_dos = 0.0       # valor de partida deliberadamente incorrecto
sobrestimacion_dos = 0.0        # valor de partida deliberadamente incorrecto

In [ ]:
uno = comprobar("C/C0 del reactor disperso de la segunda camara",
                salida_disperso_dos, 0.4483101, 1e-5)
dos = comprobar("sobrestimacion en la segunda camara", sobrestimacion_dos, 21.2282, 1e-4, "%")

### Ejercicio 4. Problema 1-6, clasificar un modelo

El Problema 1-6 del capitulo pide clasificar segun los seis criterios el modelo
de agotamiento de la zona radicular de la Seccion 1.3, que se resuelve en el
cuaderno U1-03. Ese modelo es un balance diario de un deposito, cuyo esqueleto
es mecanicista y cuyo coeficiente de cultivo es empirico, que avanza dia a dia,
que no incluye variables aleatorias, que trata toda la zona radicular como un
solo volumen, cuyo rebose introduce una conmutacion y cuyo estado es una lamina
de agua que varia de forma continua salvo en esa conmutacion.

In [ ]:
# COMPLETE: complete el diccionario con la posicion del modelo de riego en cada
# uno de los seis criterios. Los valores admitidos estan en CRITERIOS_VALIDOS.
posicion_riego = {
    "origen": "empirico",           # valor de partida deliberadamente incorrecto
    "tiempo": "estacionario",       # valor de partida deliberadamente incorrecto
    "aleatoriedad": "estocastico",  # valor de partida deliberadamente incorrecto
    "espacio": "distribuidos",      # valor de partida deliberadamente incorrecto
    "linealidad": "lineal",
    "estado": "discreto",           # valor de partida deliberadamente incorrecto
}

In [ ]:
ESPERADA_RIEGO = {"origen": "hibrido", "tiempo": "dinamico",
                  "aleatoriedad": "deterministico", "espacio": "concentrados",
                  "linealidad": "lineal", "estado": "hibrido"}
aciertos = sum(posicion_riego.get(c) == v for c, v in ESPERADA_RIEGO.items())
for criterio, esperado in ESPERADA_RIEGO.items():
    dado = posicion_riego.get(criterio)
    print(f"  {criterio:<14s} respondio {str(dado):<16s}"
          f"{'' if dado == esperado else '  revisar este criterio'}")
comprobar("criterios acertados, sobre 6", aciertos, 6, 1e-9)

### Ejercicio 5. Problema 1-24, orden observado del esquema

El Problema 1-24 pide implementar el modelo distribuido con 150, 300, 600 y 1200
celdas y estimar el orden observado. El libro afirma que al duplicar el numero
de celdas el error se reduce a la mitad, lo que confirma el orden uno del
esquema advectivo empleado.

El orden observado entre dos mallas consecutivas es el logaritmo en base dos de
la razon entre sus errores frente a la solucion analitica. Esta celda tarda cerca
de diez segundos, porque la malla de 1200 celdas es la mas costosa del cuaderno.

In [ ]:
# COMPLETE: para cada numero de celdas de MALLAS, integre el modelo distribuido,
# calcule el error relativo de la concentracion final frente a la solucion
# analitica y guarde el resultado en la lista errores. Despues estime el orden
# observado entre cada par de mallas consecutivas.
MALLAS = [150, 300, 600, 1200]
errores = [1.0, 1.0, 1.0, 1.0]     # valores de partida deliberadamente incorrectos
ordenes = [0.0, 0.0, 0.0]          # valores de partida deliberadamente incorrectos
orden_medio = 0.0                  # valor de partida deliberadamente incorrecto

In [ ]:
uno = comprobar("error relativo con 150 celdas", errores[0], 2.3164e-3, 1e-3)
dos = comprobar("error relativo con 1200 celdas", errores[-1], 2.8909e-4, 1e-3)
tres = comprobar("orden observado medio", orden_medio, 1.0, 5e-3)
if REVISAR and tres:
    print("\nEl orden observado coincide con el orden uno del esquema advectivo "
          "aguas arriba, de modo que la implementacion queda verificada.")

## 6. Problemas del capitulo

Los Problemas 1-7 a 1-10 se responden con argumentacion escrita, apoyada en la
Tabla 1.1 del libro.

- **1-7.** La curva de eficiencia de un colector solar medida entre 700 y
  1000 W/m2 es un modelo empirico, y carece de respaldo fuera de esa ventana. El
  cuaderno U1-05 cuantifica ese costo sobre un caso analogo.
- **1-8.** Para la laguna de estabilizacion, el modelo estacionario de
  parametros concentrados responde cual es la carga organica que sale en
  promedio, y el dinamico de parametros distribuidos responde como evoluciona el
  perfil ante una punta de carga, para lo cual exige la geometria, el perfil de
  velocidades y un coeficiente de dispersion.
- **1-9.** Un modelo mecanicista admite extrapolar porque los principios de
  conservacion siguen valiendo fuera del rango observado, y esa licencia cesa
  cuando una correlacion empirica de cierre, con su propio rango de validez,
  entra en el balance.
- **1-10.** La tasa media anual constante es un modelo empirico, estacionario y
  deterministico que no representa el agrupamiento de fallas en temporada de
  lluvias, y una alternativa estocastica razonable es un proceso de Poisson con
  intensidad variable en el tiempo.

Escriba sus respuestas en la celda siguiente.

### Respuestas del estudiante a los Problemas 1-7 a 1-10

*Escriba aqui. Un parrafo por problema, citando la fila de la Tabla 1.1 que
sostiene cada clasificacion.*

## Cierre

### Lista de comprobacion

Marque cada punto solo si puede hacerlo sin mirar el cuaderno.

- Redactar el modelo conceptual de un sistema con su frontera y su lista de hipotesis, antes de escribir una sola ecuacion.
- Clasificar un modelo en los seis criterios de la Tabla 1.1 y decir que consecuencia numerica trae cada posicion.
- Programar los tres modelos del Ejemplo 1.1 y explicar por que entregan respuestas distintas sin que ninguna sea incorrecta.
- Verificar un esquema numerico contra una solucion analitica y estimar su orden observado por refinamiento.

### Que revisar si algo no salio

- Si el modelo distribuido no converge al valor analitico, revise la cara de entrada, que en la condicion de Danckwerts no lleva termino dispersivo, y la cara de salida, que va sin gradiente.
- Si el orden observado sale distinto de uno, compruebe que el error se mide contra la solucion analitica y no contra la malla mas fina.
- Si la clasificacion del Ejercicio 4 no cierra, relea la Seccion 1.3 del libro, donde el modelo de riego se describe como de origen hibrido y estado continuo con conmutacion.
- Para la teoria, relea la Seccion 1.2 del libro, el Ejemplo 1.1, la Tabla 1.1 y las Figuras 1.2 y 1.3.

### Declaracion del uso de asistentes de programacion

Si empleo un asistente basado en modelos de lenguaje para resolver alguna celda, declarelo en la entrega, indique en cual y describa que prueba aplico para convencerse de que el codigo es correcto. La regla de la asignatura es que el estudiante responde por el resultado que firma, con independencia de quien escriba las lineas.